In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%load_ext memory_profiler

In [ ]:
import pandas as pd
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from src.feature_processing import *
from src.unit_proccessing import  *
from src.utils.stats_utils import *
from src.utils.chart_utils import *
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
with initialize(config_path='../configuration', version_base='1.1'):
    config = compose(config_name='main.yaml')

In [ ]:
features_class = UnitDataProcessing(config)
self = features_class

In [ ]:
df_item = features_class.df_item
df_unit = features_class.df_unit
df_unit_score = features_class.df_unit_score

In [ ]:
features_class.make_global_score()
df_unit1 = features_class.df_unit.copy()
features_class.make_global_score(combine_resp_score=True)
df_unit2 = features_class.df_unit.copy()

In [ ]:
sns.boxplot(df_unit, x='survey_version', y='unit_risk_score')
plt.xticks(rotation=90)
plt.show()

In [ ]:
df_unit1['unit_risk_score'].hist(bins=20, density=True)

In [ ]:
df_unit2['unit_risk_score'].hist(bins=20, density=True)

In [ ]:
df = df_unit.copy()
df['diff_score'] = df_unit1['unit_risk_score'] - df_unit2['unit_risk_score']
df['diff_score'].hist(density=True)

In [ ]:
less_than_0 = round(df[df['diff_score']<0]['diff_score'].count()/df['diff_score'].count()*100,1)
more_than_0 = round(df[df['diff_score']>0]['diff_score'].count()/df['diff_score'].count()*100,1)
print(less_than_0,more_than_0)
((df['diff_score']/10).astype(int)*10).value_counts()

In [ ]:
sns.scatterplot(y='unit_risk_score', x='f__number_answered', data=df_unit)
plt.show()

In [ ]:
x=df_unit[df_unit['f__number_answered']<50]
x[x.interview__id.isin(['4ca870c778f648cc986dfdafa83e3755','727f050e46fc4214a36e0afcd0c96be7'])]

In [ ]:
df_corr = df_unit.corr()

In [ ]:
features_class.make_global_score(combine_resp_score=True)
df_unit2 = features_class.df_unit.copy()
df = df_unit2
# 4. Calculate percentages

df['color'] = df['s__time_changed']>0
for col in features_class._score_columns:
    if df[col].nunique()>1:
        sns.scatterplot(y='unit_risk_score', x=col, data=df, hue='color')
        
    else:
        df[df[col] == 0]['unit_risk_score'].hist()
        df[df[col] == 1]['unit_risk_score'].hist()
        
    plt.title(f"unit_risk_score vs. {col}")
    plt.show()

In [ ]:
df.groupby(['s__gps_spatial_extreme_outlier'])['unit_risk_score'].mean()

In [ ]:
df.groupby('s__number_answered'+'_lower')['f__number_answered'].max()

In [ ]:
df[df['s__number_answered'+'_lower']<25][['f__number_answered','s__number_answered'+'_lower']]